# Исследовательский анализ данных

In [ ]:
# Импорт необходимых для работы библиотек
import pandas as pd
import numpy as np
import ast

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
def data_info(data):
""" Изучает первичную информацию о датасете """

    display(data.head())
    print('________'*10)
    display(data.info())
    print('________'*10)
    display(data.describe().T)
    print('________'*10)
    display('Количество пропусков:', data.isna().sum())
    print('________'*10)
    display('Количество явных дубликатов:', data.duplicated().sum())
    print('________'*10)
data_info(df)


In [ ]:
# Подсчет распределения вакансий по источникам
counts = df['data_source'].value_counts()
percentages = df['data_source'].value_counts(normalize=True) * 100

plt.figure(figsize=(10, 8))
ax = counts.plot(kind='bar')

for i, source in enumerate(counts.index):
    ax.text(
        i,
        counts[source],
        f"{counts[source]}\n({percentages[source]:.1f}%)",
        ha='center',
        va='bottom'
    )

plt.title('Распределение вакансий по источникам')
plt.xlabel('Источник данных')
plt.ylabel('Количество вакансий')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Исследование распределений уровня зарплаты по вакансиям
# Зарплата
salary_cols = ['salary_from', 'salary_to', 'salary']
for col in salary_cols:
    plt.figure(figsize=(14, 5))

    # Гистограмма с правилом Стерджеса
    plt.subplot(1, 2, 1)
    sns.histplot(df[col].dropna(), bins='sturges', kde=True)
    plt.title(f'Гистограмма {col} (правило Стерджеса)')
    plt.xlabel('Зарплата')
    plt.ylabel('Количество вакансий')

    # Боксплот
    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[col].dropna())
    plt.title(f'Боксплот {col}')
    plt.xlabel('Зарплата')

    plt.tight_layout()
    plt.show()


In [ ]:
# Распределение средней зарплаты по региону
plt.figure(figsize=(14, 5))
sns.histplot(df['salary'].dropna(), bins='sturges', kde=True)
plt.title('Распределение средней зарплаты по региону')
plt.xlabel('Зарплата')
plt.ylabel('Количество вакансий')

plt.show()


# Очистка и подготовка данных к дальнейшей работе (ключевые моменты)

In [ ]:
# Приведение данных к необходимым форматам, заполнение пропусков
df_vacancy['employer_inn'] = df_vacancy['employer_inn'].astype('Int64').astype('string')
df_vacancy['employer_ogrn'] = df_vacancy['employer_ogrn'].astype('Int64').astype('string')

df_vacancy.loc[df_vacancy['employer_industry_name'].isna(), 'employer_industry_id'] = np.nan

df_vacancy['employer_industry_id'] = df_vacancy['employer_industry_id'].astype('Int64')
df_vacancy = df_vacancy.copy()

In [ ]:
def parse_okved_string(val):
    """Парсит строку вида [значение1, значение2, значение3]"""
    if not isinstance(val, str):
        return val if isinstance(val, list) else []

    val = val.strip()
    if val in ['', '[]', 'nan', 'NaN', '<NA>']:
        return []

    content = val[1:-1] if val.startswith('[') and val.endswith(']') else val

    if not content.strip():
        return []

    items = []
    for item in content.split(','):
        item = item.strip()
        item = item.strip('"\'')
        if item:  
            items.append(item)

    return items
for col in ['okved_ids', 'okved_id_names', 'okved_sections', 'okved_section_names']:
    df_vacancy[col] = df_vacancy[col].apply(parse_okved_string)

business_size_mapping = {
    'small': 'small',
    'middle': 'middle',
    'large': 'large',
    'micro': 'small',
    'big': 'large'
}
df_vacancy['employer_business_size'] = df_vacancy['employer_business_size'].map(business_size_mapping)



# Очистка датасета от вакансий, не относящихся к нефтехимической отрасли

In [ ]:
# Отбор значений из столбца role_name_dasboard, не относящихся к рабочим нефтехимическим профессиям
petrochemical_not = [
    'Юрист',
    'Юрисконсульт',
    'Директор юридического департамента (CLO)',
    'Специалист технической поддержки',
    'Водитель',
    'Грузчик',
    'Уборщица, уборщик',
    'Дворник',
    'Разнорабочий',
    'Подсобный рабочий',
    'Повар, пекарь, кондитер',
    'Упаковщик, комплектовщик',
    'Продавец-консультант, продавец-кассир',
    'Директор магазина, директор сети магазинов',
    'Администратор магазина, администратор торгового зала',
    'Товаровед',
    'Мерчандайзер',
    'Швея, портной, закройщик',
    'Маляр, штукатур',
    'Печатник',
    'Охранник',
    'Автослесарь, автомеханик',
    'Автомойщик',
    'Стропальщик',
    'Резчик металла на ножницах и прессах',
    'Резчик стекла',
    'Штамповщик',
    'Термист',
    'Бетонщик',
    'Газорезчик',
    'Монтер пути',
    'Осмотрщик-ремонтник вагонов',
    'Воспитатель, няня',
    'Учитель, преподаватель, педагог',
    'Медицинская сестра, медицинский брат',
    'Врач',
    'Ветеринарный врач',
    'Санитарка/санитар',
    'Фармацевт',
    'Заведующий аптекой',
    'Медицинский представитель',
    'Фитнес-тренер, инструктор тренажерного зала',
    'Полицейский',
    'Кинолог',
    'Электромонтер оперативно-выездной бригады',
    'Агроном',
    'Зоотехник',
    'Животновод',
    'Главный рыбовод',
    'Садовник',
    'Лесовод',
    'Геодезист',
    'Кадастровый инженер',
    'Архитектор',
    'Дизайнер, художник',
    'Фотограф, ретушер',
    'Видеооператор, видеомонтажер',
    'Копирайтер, редактор, корректор',
    'SMM-менеджер, контент-менеджер',
    'Event-менеджер',
    'PR-менеджер',
    'Торговый представитель',
    'Официант, бармен, бариста',
    'Кассир-операционист',
    'Оператор стиральных машин',
    'Оператор call-центра, специалист контактного центра',
    'Документовед',
    'Переводчик',
    'Геолог',
    'Авиационный техник',
    'Инженер по вентиляции и кондиционированию',
    'Инженер по обследованию зданий и сооружений',
    'Руководитель строительного',
    'Руководитель строительного проекта',
    'Прораб, мастер СМР',
    'Менеджер по строительству',
    'Специалист службы безопасности',
    'Координатор отдела продаж',
    'Супервайзер',
    'Менеджер по продажам, менеджер по работе с клиентами',
    'Экономист',
    'Программист, разработчик',
    'Кладовщик',
    'Аналитик',
    'Бухгалтер',
    'Маркетолог-аналитик'
]


In [ ]:
df_all_clean = df_all[
    ~df_all['role_name_dashboard'].isin(petrochemical_not)
]
df_tmp = df_all_clean.copy()

df_tmp['okved_ids'] = df_tmp['okved_ids'].apply(
    lambda x: ';'.join(map(str, x)) if isinstance(x, list) else x
)

In [ ]:
# Отбор уникальных компаний для отбора нефтехимических
df_companies = (
    df_tmp[['employer_name', 'employer_inn', 'okved_ids']]
    .drop_duplicates()
    .reset_index(drop=True)
)